# 00 — Data Load

Downloads the **Diabetes Health Indicators Dataset** (CDC BRFSS 2015) from Kaggle via `kagglehub` and places the CSV at the exact path `config.py` expects (`data/diabetes_binary_health_indicators_BRFSS2015.csv`). Run this notebook first, before `01_eda.ipynb`.

**Requirements:** a free Kaggle account and API credentials. On first run, `kagglehub` will prompt for your Kaggle username and API key (or read them from `~/.kaggle/kaggle.json` / the `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables if already configured). Get a key from https://www.kaggle.com/settings → API → "Create New Token".

In [1]:
import shutil
from pathlib import Path

import kagglehub

from config import RAW_CSV, DATA_DIR

DATASET_SLUG = "alexteboul/diabetes-health-indicators-dataset"
SOURCE_FILENAME = "diabetes_binary_health_indicators_BRFSS2015.csv"

## Download the dataset

`kagglehub` caches the download under a local cache directory and returns the path to it — it will not re-download if it's already cached.

In [2]:
cache_path = Path(kagglehub.dataset_download(DATASET_SLUG))
print(f"Dataset cached at: {cache_path}")
print("Files found:")
for f in cache_path.iterdir():
    print(f" - {f.name}")

Dataset cached at: C:\Users\ardao\.cache\kagglehub\datasets\alexteboul\diabetes-health-indicators-dataset\versions\1
Files found:
 - diabetes_012_health_indicators_BRFSS2015.csv
 - diabetes_binary_5050split_health_indicators_BRFSS2015.csv
 - diabetes_binary_health_indicators_BRFSS2015.csv


## Copy the CSV into `data/`

Copies (does not move) the file so the cache stays intact, placing it exactly where `config.RAW_CSV` points so the rest of the pipeline (`01_eda.ipynb` onward) can find it without changes.

In [3]:
source_file = cache_path / SOURCE_FILENAME

if not source_file.exists():
    candidates = list(cache_path.glob("*.csv"))
    if len(candidates) == 1:
        source_file = candidates[0]
    else:
        raise FileNotFoundError(
            f"Expected {SOURCE_FILENAME} in {cache_path}, found: {[c.name for c in candidates]}"
        )

DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(source_file, RAW_CSV)
print(f"Copied dataset to: {RAW_CSV}")

Copied dataset to: C:\Users\ardao\Desktop\Projects\Bil476\data\diabetes_binary_health_indicators_BRFSS2015.csv


## Verify

In [4]:
import pandas as pd

df = pd.read_csv(RAW_CSV)
print(f"Shape: {df.shape}")
df.head()

Shape: (253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
